# `garden audit` on public data

The Sullivan–Timmermann–White (1999) universe of moving-average crossover rules, on SPY
daily adjusted closes (`data/SPY_daily.csv`, 1993-01-29 to 2026-09-18).

Short window ∈ {5, 10, 20, 50}, long ∈ {50, 100, 200}, short < long, each with and without a
1% band, each long-only and long-short: **44 rules**. The grid is written down before any
return is computed, so the menu is **data-oblivious** and the Reality Check is valid for the
trial count as logged — no re-execution and no class enumeration needed.

A 1% band on windows this far apart leaves every rule in the market at least ~50% of days
(minimum 4,109 distinct active periods against the audit's `SUPPORT_MIN = 50`), so the
sparse-position pathology of SCOPE.md §11 does not apply at this sample length. If it ever
did, the audit would return `DEGENERATE` rather than a misleading `FAIL`.

Three audits are run below. They differ only in **what null they test**, and they disagree —
which is the point.

In [ ]:
import numpy as np, pandas as pd
from estimator.bootstrap import sharpe
from garden.audit import audit
from garden.preflight import preflight
from garden.transcript import Transcript

PPY, WARMUP, BAND = 252, 200, 0.01
px = (pd.read_csv("../data/SPY_daily.csv", parse_dates=["Date"])
        .sort_values("Date")["Adj Close"].to_numpy(float))
r = np.diff(np.log(px))          # daily log returns
price = px[1:]                   # the close that decides the next day's return
ann, bh = np.sqrt(PPY), r[WARMUP:]                    # buy-and-hold, the benchmark
print(f"{len(px)} bars, {len(r)} returns")

In [ ]:
def ma(a, w):
    c = np.concatenate([[0.0], np.cumsum(a)])
    m = np.full(len(a), np.nan); m[w - 1:] = (c[w:] - c[:-w]) / w
    return m

# Fixed before a single return is looked at. That is what makes the menu oblivious.
pairs = [(f, s) for f in (5, 10, 20, 50) for s in (50, 100, 200) if f < s]
rules = [(f, s, b, lo) for f, s in pairs for b in (0.0, BAND) for lo in (False, True)]

cols, ids = [], []
for f, s, b, lo in rules:
    mf, ms = ma(price, f), ma(price, s)
    pos = np.zeros(len(r))
    pos[mf > ms * (1 + b)] = 1.0
    pos[mf < ms * (1 - b)] = -1.0
    if lo: pos = np.maximum(pos, 0.0)
    cols.append(pos[WARMUP - 1:-1] * r[WARMUP:])          # yesterday's signal, today's return
    ids.append(f"ma_{f}_{s}_{'band' if b else 'noband'}_{'long' if lo else 'ls'}")

R = np.column_stack(cols)
sr = sharpe(R, axis=0, annualization=ann)
print(f"{R.shape[1]} rules over {R.shape[0]} periods; best = {ids[int(np.argmax(sr))]} "
      f"at Sharpe {sr.max():.4f}")

In [ ]:
pf = preflight(n_specs=R.shape[1], n_periods=R.shape[0], reference_sharpe=1.0, periods_per_year=PPY)
s0 = pf.scenarios[0]
print(f"PREFLIGHT  {pf.n_specs} specs x {pf.n_periods} periods")
print(f"  critical value {s0.critical_value:.4f}   power {s0.power:.4f}   "
      f"required Sharpe {s0.required_sharpe:.4f}")
for line in pf.reasons: print("  -", line)

In [ ]:
def show(tag, RR, II, null_sentence):
    s = sharpe(RR, axis=0, annualization=ann)
    v = audit(Transcript(RR, II, II[int(np.argmax(s))], menu_kind="oblivious",
                         periods_per_year=PPY))
    print(f"=== {tag} -> {v.status} ===")
    print(f"  null tested: {null_sentence}")
    print(f"  submitted {v.submitted} (rank {v.submitted_rank} of {v.n_trials})")
    print(f"  in-sample {v.sr_reported:+.4f}   null max mean {v.null_max_mean:.4f}   "
          f"deflated {v.sr_deflated:+.4f}")
    print(f"  critical value {v.critical_value:.4f}   p {v.p_value:.4f}   "
          f"block {v.block_length}   degenerate share {v.degenerate_share:.3f}")
    for line in v.reasons: print("   -", line)
    print()
    return v

ls = [i for i, n in enumerate(ids) if n.endswith("_ls")]

v0 = show("(0) 44 rules, as traded", R, ids,
          "zero mean return. Does the best rule beat not trading at all, once the 44-way "
          "search is priced in? It does not ask whether the rule beats owning the index.")

va = show("(a) 44 rules, in excess of buy-and-hold", R - bh[:, None], ids,
          "zero mean EXCESS return. Does the best rule beat holding SPY over the same days, "
          "which strips out the equity premium that (0) leaves in?")

vb = show("(b) 22 long-short rules, as their own class", R[:, ls], [ids[i] for i in ls],
          "zero mean return again, on the subset allowed to go short. That lifts the "
          "no-shorting constraint; it does not remove market exposure, since every one of "
          "these rules is net long on average (the winner, +0.54).")

## Why the three disagree

The audit's null is **zero mean return**, not buy-and-hold, and it corrects for search breadth
only — as its own last reason says. Most of this grid is long-only on an index that rose
thirtyfold, so (0) can clear a breadth-corrected bar on equity beta alone, with no timing edge
whatever. Audit (a) re-runs the identical 44-rule search against the benchmark instead of
against zero, and that is the comparison that answers "is there a timing edge?".

Audit (b) is not a beta-free control. Going short is permitted, not required: all 22 long-short
rules hold a positive average position, so (b) tests a smaller, differently-constrained class
against the same zero null — not the market-relative question (a) asks.

The cell below prices the exposure directly.

In [ ]:
j = int(np.argmax(sr))
lo_ = [i for i, n in enumerate(ids) if n.endswith("_long")]
print(f"buy-and-hold SPY        {sharpe(bh, annualization=ann):.4f}")
print(f"best rule {ids[j]:<22} {sr[j]:.4f}")
print(f"  rule minus buy-and-hold {sharpe(R[:, j] - bh, annualization=ann):+.4f}")
print(f"  days with non-zero P&L  {(R[:, j] != 0).mean():.3f}")   # a floor on time in market
print(f"long-short rules  max {sr[ls].max():.4f}  median {np.median(sr[ls]):.4f}")
print(f"long-only rules   max {sr[lo_].max():.4f}  median {np.median(sr[lo_]):.4f}")